In [17]:
import tensorflow as tf
import pandas as pd
from keras import layers
from sklearn.metrics import classification_report

train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df = pd.read_csv("../data/processed/test.csv")

vectorizer = layers.TextVectorization(max_tokens=20_000, output_sequence_length=128)
vectorizer.adapt(train_df["text"].astype(str).values)

X_train = vectorizer(train_df["text"].astype(str).values)
X_val = vectorizer(val_df["text"].astype(str).values)
X_test = vectorizer(test_df["text"].astype(str).values)

train_labels = train_df["label"].values
val_labels = val_df["label"].values
test_labels = test_df["label"].values

In [ ]:
cnn_model = tf.keras.Sequential([
    layers.Embedding(20_000, 32),
    layers.Conv1D(32, 5, activation="relu", padding="same"),
    layers.MaxPooling1D(2),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])

rnn_model = tf.keras.Sequential([
    layers.Embedding(20_000, 32),
    layers.SimpleRNN(32),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])

In [ ]:
cnn_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
cnn_model.fit(X_train, train_labels, epochs=5, batch_size=32, validation_data=(X_val, val_labels), verbose=1)

rnn_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
rnn_model.fit(X_train, train_labels, epochs=5, batch_size=32, validation_data=(X_val, val_labels), verbose=1)

Training CNN...
Epoch 1/5
9601/9601 ━━━━━━━━━━━━━━━━━━━━ 37s 4ms/step - accuracy: 0.9333 - loss: 0.2007 - val_accuracy: 0.9426 - val_loss: 0.1889
Epoch 2/5
9601/9601 ━━━━━━━━━━━━━━━━━━━━ 36s 4ms/step - accuracy: 0.9522 - loss: 0.1377 - val_accuracy: 0.9453 - val_loss: 0.1636
Epoch 3/5
9601/9601 ━━━━━━━━━━━━━━━━━━━━ 36s 4ms/step - accuracy: 0.9659 - loss: 0.0968 - val_accuracy: 0.9490 - val_loss: 0.1484
Epoch 4/5
9601/9601 ━━━━━━━━━━━━━━━━━━━━ 36s 4ms/step - accuracy: 0.9761 - loss: 0.0686 - val_accuracy: 0.9486 - val_loss: 0.1531
Epoch 5/5
9601/9601 ━━━━━━━━━━━━━━━━━━━━ 37s 4ms/step - accuracy: 0.9823 - loss: 0.0507 - val_accuracy: 0.9454 - val_loss: 0.1593

CNN Test Accuracy: 0.9473
Training RNN...
Epoch 1/5
9601/9601 ━━━━━━━━━━━━━━━━━━━━ 85s 9ms/step - accuracy: 0.8894 - loss: 0.3229 - val_accuracy: 0.8861 - val_loss: 0.3527
Epoch 2/5
9601/9601 ━━━━━━━━━━━━━━━━━━━━ 85s 9ms/step - accuracy: 0.8915 - loss: 0.3052 - val_accuracy: 0.8980 - val_loss: 0.2848
Epoch 3/5
9601/9601 ━━━━━━━━━━━

In [ ]:
cnn_acc = cnn_model.evaluate(X_test, test_labels)[1]
print(f"CNN Accuracy: {cnn_acc}")
cnn_probs = cnn_model.predict(X_test).flatten()
cnn_preds = (cnn_probs >= 0.5).astype(int)
print(classification_report(test_labels, cnn_preds, target_names=["Human", "LLM"]))

rnn_acc = rnn_model.evaluate(X_test, test_labels)[1]
print(f"RNN Test Accuracy: {rnn_acc}")
rnn_probs = rnn_model.predict(X_test).flatten()
rnn_preds = (rnn_probs >= 0.5).astype(int)
print(classification_report(test_labels, rnn_preds, target_names=["Human", "LLM"]))


CNN Test Accuracy: 0.9473
              precision    recall  f1-score   support

       Human       0.79      0.73      0.76      7500
          AI       0.97      0.98      0.97     58333

    accuracy                           0.95     65833
   macro avg       0.88      0.85      0.86     65833
weighted avg       0.95      0.95      0.95     65833

ROC-AUC: 0.9517

RNN Test Accuracy: 0.8888
              precision    recall  f1-score   support

       Human       0.53      0.20      0.29      7500
          AI       0.90      0.98      0.94     58333

    accuracy                           0.89     65833
   macro avg       0.72      0.59      0.61     65833
weighted avg       0.86      0.89      0.87     65833

ROC-AUC: 0.7481
